# Weighted vs Unweighted Reconstruction Comparison
Side-by-side comparison of reconstructions from weighted and unweighted models

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
from tqdm.auto import tqdm

# Runtime reload
import importlib
import models.autoencoder.bottleneck_models
importlib.reload(models.autoencoder.bottleneck_models)

from data.data_ingestion import collect_files, generate_dataframe
from data.dataloader import create_dataloaders
from models.autoencoder.bottleneck_models import (
    BottleneckEncoder,
    ComplexBottleneckDeconvDecoder,
    BottleneckAE,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Data Setup

In [ ]:
root_dir = Path(".")
images_dir = root_dir / "data" / "Images"
mask_path = root_dir / "data" / "masks" / "rmask_ICV.nii"

# Configuration
data_dir = "data/Images"
mask_path = "data/masks/rmask_ICV.nii"
batch_size = 4
unweighted_output_dir = "output/Experiments/BottleneckDeconvTraining"
weighted_output_dir = "output/Experiments/BottleneckDeconvWeightedTraining"
comparison_output_dir = "output/Experiments/WeightedUnweightedComparison"

# Create output directory
os.makedirs(comparison_output_dir, exist_ok=True)

In [ ]:
# Prepare data
print("Collecting files...")
included_files, excluded_files = collect_files(data_dir)
print(f"Found {len(included_files)} valid files, excluded {len(excluded_files)} files")

# Generate dataframe
print("Generating dataframe...")
df = generate_dataframe(included_files)

# Create dataloaders
print("Creating dataloaders...")
train_loader, val_loader = create_dataloaders(
    df, batch_size=batch_size, train_split=0.8, 
    on_demand=True, mask_path=mask_path,
    num_workers=2
)

print(f"Data preparation complete. Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}")

## Model Setup

In [ ]:
target_shape = (64, 128, 128)

def build_model(latent_dim=128):
    """Build ComplexBottleneckDeconvDecoder model"""
    model = BottleneckAE(
        BottleneckEncoder(initial_filters=4, latent_dim=latent_dim, bottleneck_shape=(1,1,1)),
        ComplexBottleneckDeconvDecoder(latent_dim=latent_dim, target_shape=target_shape),
    )
    return model.to(device)

## Load All Models

In [ ]:
# Load all unweighted trained models from checkpoints
unweighted_models = {}
deconv_latent_dims = [512, 256, 128, 64, 32]

for latent_dim in deconv_latent_dims:
    model_name = f"deconv_lat{latent_dim}"
    checkpoint_path = os.path.join(unweighted_output_dir, model_name, f"{model_name}_best.pth")
    
    if os.path.exists(checkpoint_path):
        print(f"Loading {model_name} from {checkpoint_path}...")
        model = build_model(latent_dim=latent_dim)
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        unweighted_models[model_name] = model.eval()
        print(f"✓ Loaded {model_name}")
    else:
        print(f"✗ Checkpoint not found for {model_name}")

print(f"\nLoaded {len(unweighted_models)} unweighted models")

In [ ]:
# Load all weighted trained models from checkpoints
weighted_models = {}

for latent_dim in deconv_latent_dims:
    model_name = f"deconv_weighted_lat{latent_dim}"
    checkpoint_path = os.path.join(weighted_output_dir, model_name, f"{model_name}_best.pth")
    
    if os.path.exists(checkpoint_path):
        print(f"Loading {model_name} from {checkpoint_path}...")
        model = build_model(latent_dim=latent_dim)
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        weighted_models[model_name] = model.eval()
        print(f"✓ Loaded {model_name}")
    else:
        print(f"✗ Checkpoint not found for {model_name}")

print(f"\nLoaded {len(weighted_models)} weighted models")

## Get Batch for Visualization

In [ ]:
# Get a batch for reconstruction visualization
batch = next(iter(val_loader))
inp = batch['volume'].to(device)
labels = batch['label']

print(f"Input batch shape: {inp.shape}")
print(f"Labels: {labels}")

# Get reconstructions from all unweighted models
unweighted_reconstructions = {}
with torch.no_grad():
    for model_name, model in unweighted_models.items():
        out = model(inp)
        unweighted_reconstructions[model_name] = out

# Get reconstructions from all weighted models
weighted_reconstructions = {}
with torch.no_grad():
    for model_name, model in weighted_models.items():
        out = model(inp)
        weighted_reconstructions[model_name] = out

## Side-by-Side Comparison - Axial Slices

In [ ]:
# Visualize side-by-side comparisons for each sample in batch
for pIdx in range(min(4, inp.shape[0])):
    fig = plt.figure(figsize=(24, 20))
    
    # Loop through each latent dimension
    for col_idx, latent_dim in enumerate(deconv_latent_dims):
        unweighted_model_name = f"deconv_lat{latent_dim}"
        weighted_model_name = f"deconv_weighted_lat{latent_dim}"
        
        unweighted_recon = unweighted_reconstructions[unweighted_model_name]
        weighted_recon = weighted_reconstructions[weighted_model_name]
        
        # Original image
        ax = plt.subplot(len(deconv_latent_dims), 3, col_idx * 3 + 1)
        ax.imshow(inp[pIdx, 0, 32, 10:-10, 10:-10].cpu(), cmap='gray', vmin=-0., vmax=4.)
        if col_idx == 0:
            ax.set_ylabel(f"Sample {pIdx + 1}\n({labels[pIdx]})\nOriginal", fontsize=12, fontweight='bold')
        ax.set_title(f"Lat Dim: {latent_dim}", fontsize=11, fontweight='bold')
        ax.axis('off')
        
        # Unweighted reconstruction
        ax = plt.subplot(len(deconv_latent_dims), 3, col_idx * 3 + 2)
        ax.imshow(unweighted_recon[pIdx, 0, 32, 10:-10, 10:-10].cpu(), cmap='gray', vmin=-0., vmax=4.)
        if col_idx == 0:
            ax.set_ylabel(f"Unweighted\nReconstruction", fontsize=12, fontweight='bold')
        ax.axis('off')
        
        # Weighted reconstruction
        ax = plt.subplot(len(deconv_latent_dims), 3, col_idx * 3 + 3)
        ax.imshow(weighted_recon[pIdx, 0, 32, 10:-10, 10:-10].cpu(), cmap='gray', vmin=-0., vmax=4.)
        if col_idx == 0:
            ax.set_ylabel(f"Weighted\nReconstruction", fontsize=12, fontweight='bold')
        ax.axis('off')
    
    plt.suptitle(f"Weighted vs Unweighted Reconstruction Comparison - Sample {pIdx + 1} ({labels[pIdx]}) - Axial View", 
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(os.path.join(comparison_output_dir, f"comparison_axial_sample_{pIdx+1}.png"), dpi=150, bbox_inches='tight')
    plt.show()

print(f"Axial comparison visualizations saved to {comparison_output_dir}")

## Side-by-Side Comparison - Coronal Slices

In [ ]:
# Visualize side-by-side comparisons for each sample in batch (coronal view)
for pIdx in range(min(4, inp.shape[0])):
    fig = plt.figure(figsize=(24, 20))
    
    # Loop through each latent dimension
    for col_idx, latent_dim in enumerate(deconv_latent_dims):
        unweighted_model_name = f"deconv_lat{latent_dim}"
        weighted_model_name = f"deconv_weighted_lat{latent_dim}"
        
        unweighted_recon = unweighted_reconstructions[unweighted_model_name]
        weighted_recon = weighted_reconstructions[weighted_model_name]
        
        # Original image
        ax = plt.subplot(len(deconv_latent_dims), 3, col_idx * 3 + 1)
        ax.imshow(inp[pIdx, 0, :, 50, 10:-10].cpu(), cmap='viridis', vmin=-0., vmax=4.)
        ax.set_ylim(0, 64)
        if col_idx == 0:
            ax.set_ylabel(f"Sample {pIdx + 1}\n({labels[pIdx]})\nOriginal", fontsize=12, fontweight='bold')
        ax.set_title(f"Lat Dim: {latent_dim}", fontsize=11, fontweight='bold')
        ax.axis('off')
        
        # Unweighted reconstruction
        ax = plt.subplot(len(deconv_latent_dims), 3, col_idx * 3 + 2)
        ax.imshow(unweighted_recon[pIdx, 0, :, 50, 10:-10].cpu(), cmap='viridis', vmin=-0., vmax=4.)
        ax.set_ylim(0, 64)
        if col_idx == 0:
            ax.set_ylabel(f"Unweighted\nReconstruction", fontsize=12, fontweight='bold')
        ax.axis('off')
        
        # Weighted reconstruction
        ax = plt.subplot(len(deconv_latent_dims), 3, col_idx * 3 + 3)
        ax.imshow(weighted_recon[pIdx, 0, :, 50, 10:-10].cpu(), cmap='viridis', vmin=-0., vmax=4.)
        ax.set_ylim(0, 64)
        if col_idx == 0:
            ax.set_ylabel(f"Weighted\nReconstruction", fontsize=12, fontweight='bold')
        ax.axis('off')
    
    plt.suptitle(f"Weighted vs Unweighted Reconstruction Comparison - Sample {pIdx + 1} ({labels[pIdx]}) - Coronal View", 
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(os.path.join(comparison_output_dir, f"comparison_coronal_sample_{pIdx+1}.png"), dpi=150, bbox_inches='tight')
    plt.show()

print(f"Coronal comparison visualizations saved to {comparison_output_dir}")

## Difference Maps - Axial

In [ ]:
# Visualize difference maps between weighted and unweighted reconstructions
for pIdx in range(min(4, inp.shape[0])):
    fig = plt.figure(figsize=(24, 20))
    
    # Loop through each latent dimension
    for col_idx, latent_dim in enumerate(deconv_latent_dims):
        unweighted_model_name = f"deconv_lat{latent_dim}"
        weighted_model_name = f"deconv_weighted_lat{latent_dim}"
        
        unweighted_recon = unweighted_reconstructions[unweighted_model_name]
        weighted_recon = weighted_reconstructions[weighted_model_name]
        
        # Difference map
        diff = (weighted_recon - unweighted_recon)[pIdx, 0, 32, 10:-10, 10:-10].cpu()
        
        # Unweighted reconstruction
        ax = plt.subplot(len(deconv_latent_dims), 2, col_idx * 2 + 1)
        ax.imshow(unweighted_recon[pIdx, 0, 32, 10:-10, 10:-10].cpu(), cmap='gray', vmin=-0., vmax=4.)
        if col_idx == 0:
            ax.set_ylabel(f"Sample {pIdx + 1}\n({labels[pIdx]})\nUnweighted", fontsize=12, fontweight='bold')
        ax.set_title(f"Lat Dim: {latent_dim}", fontsize=11, fontweight='bold')
        ax.axis('off')
        
        # Difference map
        ax = plt.subplot(len(deconv_latent_dims), 2, col_idx * 2 + 2)
        im = ax.imshow(diff, cmap='RdBu_r', vmin=-2., vmax=2.)
        if col_idx == 0:
            ax.set_ylabel(f"Difference\n(Weighted - Unweighted)", fontsize=12, fontweight='bold')
        ax.axis('off')
        if col_idx == len(deconv_latent_dims) - 1:
            cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label('Difference', rotation=270, labelpad=15)
    
    plt.suptitle(f"Difference Maps - Sample {pIdx + 1} ({labels[pIdx]}) - Axial View", 
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(os.path.join(comparison_output_dir, f"difference_axial_sample_{pIdx+1}.png"), dpi=150, bbox_inches='tight')
    plt.show()

print(f"Difference map visualizations saved to {comparison_output_dir}")